# Durable execution: retries, timers, compensation, and state ownership

**Claim cluster:** `C009`, `C010`, `C011`, `C012`, `C013`, `C014`, `C015`, `C016`, `C036`, `C080`  
**Source of truth:** `data/AutomationResearch.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import os
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# Option A topology: this wave reads one dedicated Stage 2 database. Point
# viz_core at it before import -- it resolves the path once at import time --
# so every fetch in this kernel uses the same file.
os.environ["SMARTHOME_DB"] = str(ROOT / "data" / "AutomationResearch.duckdb")

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in this notebook reads only from data/AutomationResearch.duckdb (read-only).
CLAIM_IDS = ('C009', 'C010', 'C011', 'C012', 'C013', 'C014', 'C015', 'C016', 'C036', 'C080')
assert viz.DEFAULT_DB_PATH.endswith("AutomationResearch.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

C009-C015 compare Temporal's durable-runtime capabilities (state, retries, timers, Saga compensation) with LangGraph's checkpointer-based state. C036 records a silent-corruption incident (15,000 profiles over 90 days) that underscores why durability matters. C080 notes both Temporal and LangGraph emphasize stateful, long-running execution.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Durable execution capabilities mapped to claim text

The claims describe four capabilities (state, retries, timers, compensation) and how each runtime implements them. The figure maps claim readings to capabilities.

In [ ]:
# Capability mapping from claim text
CAPS = ["State ownership", "Automatic retries", "Durable timers",
        "Saga compensation", "Checkpoint persistence"]
# Readings taken from claim text manually (authoring discipline)
READINGS = {
    "C009": ("State ownership", 1),
    "C010": ("State ownership", 0.5),
    "C011": ("Automatic retries", 1),
    "C012": ("Automatic retries", 0.5),
    "C013": ("Durable timers", 1),
    "C014": ("Durable timers", 0.5),
    "C015": ("Saga compensation", 1),
    "C016": ("Saga compensation", 0),
}

_rows, _texts = [], []
for cap in CAPS:
    _row, _row_text = [], []
    for cid in ["C009","C010","C011","C012","C013","C014","C015","C016"]:
        cap_read, status = READINGS.get(cid, (None, None))
        # Only colour cells where this claim applies to this cap
        if cap_read == cap:
            _row.append(status)
            _row_text.append(cid)
        else:
            _row.append(None)
            _row_text.append("")
    _rows.append(_row)
    _texts.append(_row_text)

fig = go.Figure(go.Heatmap(
    z=[[v if v is not None else -1 for v in row] for row in _rows],
    x=["C009","C010","C011","C012","C013","C014","C015","C016"],
    y=CAPS,
    zmin=-1, zmax=1,
    colorscale=[[0.0,"#e8b6bb"],[0.5,"#f3e2b8"],[1.0,"#bcd9c4"]],
    text=_texts,
    texttemplate="%{text}",
    customdata=[[f"{cid}: {READINGS[cid][1]}" if cid in READINGS else "" for cid in ["C009","C010","C011","C012","C013","C014","C015","C016"]] for _ in CAPS],
    hovertemplate="%{customdata}<extra></extra>",
))
fig.update_layout(
    title="Durable execution claim readings (1 = present / qualified, 0 = absent / slower, empty = not claimed)",
    height=320, xaxis_side="top", yaxis_autorange="reversed")
fig.show()

## Supporting context

### Metrics and evidence behind reliability claims

M10-M11 (Temporal base plans: $100, $500/month), M25 (Claude managed agents $0.08/session-hour), M27 (self-hosted Temporal infra $480-$790/month), and M28 (Temporal actions $50/million) provide cost context for durable execution. M36 (human review rate 20%) connects reliability to operational labor.

In [ ]:
display(HTML("<h4>Reliability-related metrics (cost + operational)</h4>"))
_rel = viz.metrics_by_id(["M10","M11","M25","M27","M28","M36","M37","M38"])
show(_rel[["local_id","metric_name","value","unit","confidence"]].rename(
    columns={"local_id":"m","metric_name":"name","confidence":"conf"}))

## Uncertainty and gaps

### Reliability claim limits

- **C010 says LangGraph does not own durable execution** — a documented fact with high confidence, but it describes an absence rather than a capability.
- **C036 (silent corruption of 15,000 profiles) is documented fact (high)** but applies to one RPA bot incident, not to all durable runtimes.
- **C012 (node-level retries require idempotent side effects)** is a documented fact that limits replay, not a guarantee.

In [ ]:
# Incident claim C036 and its metric links
_incident = claims.loc[claims["local_id"] == "C036", ["local_id","claim_type","confidence","claim_text"]]
display(HTML("<h4>Incident claim C036 (silent corruption, 90 days)</h4>"))
show(_incident.rename(columns={"local_id":"claim","claim_type":"type","confidence":"conf"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Durable-capability mapping is authored from claim text (C009-C016) and shown above for audit; metrics M10-M11, M25, M27, M28, M36-M38 come from `viz.metrics_by_id`. Incident claim C036 is verified via `viz.fetch_claims`.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()